# Training Model Klasifikasi dari Data PDF
## Membaca PDF surat dari folder dataset, preprocess, lalu train model

**Peneliti:** Rivaldo Janter Tampubolon (221011402289)

---

### Struktur folder dataset:
```
dataset/
  Purchase Order/
    surat_001.pdf
    surat_002.pdf
  Invoice/
    inv_001.pdf
    inv_002.pdf
  Surat Penawaran/
    ...
  Kontrak/
    ...
  Nota Dinas/
    ...
  MoU/
    ...
  Lainnya/
    ...
```

**Cara pakai:**
1. Buat folder sesuai nama kategori di atas
2. Masukkan PDF surat ke masing-masing folder
3. Jalankan semua cell dari atas ke bawah

In [ ]:
# 1. IMPORT LIBRARY
import os, re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import pdfplumber
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.metrics import (classification_report, confusion_matrix,
    accuracy_score, precision_score, recall_score, f1_score)
from Sastrawi.Stemmer.StemmerFactory import StemmerFactory
from Sastrawi.StopWordRemover.StopWordRemoverFactory import StopWordRemoverFactory

print('Library berhasil diimport!')

## 2. Baca PDF dari Folder Dataset

In [ ]:
# Path ke folder dataset
DATASET_DIR = '../dataset'

# Mapping nama folder ke arah dokumen
# Folder "Masuk" = Surat Masuk, selain itu = Surat Keluar
ARAH_MAP = {
    'Masuk': 'Masuk',
    'Surat Masuk': 'Masuk',
}

def extract_text_from_pdf(pdf_path):
    """Ekstrak teks dari satu file PDF."""
    text = ''
    try:
        with pdfplumber.open(pdf_path) as pdf:
            for page in pdf.pages:
                t = page.extract_text()
                if t:
                    text += t + '\n'
    except Exception as e:
        print(f'  [ERROR] {os.path.basename(pdf_path)}: {e}')
    return text.strip()

# Baca semua PDF
data = []
print('Membaca PDF dari folder dataset...')
print('=' * 60)

for folder_name in os.listdir(DATASET_DIR):
    folder_path = os.path.join(DATASET_DIR, folder_name)
    if not os.path.isdir(folder_path):
        continue

    # Nama folder = jenis dokumen
    jenis = folder_name

    # Tentukan arah dari nama folder atau subfolder
    arah_default = ARAH_MAP.get(folder_name, 'Keluar')

    pdf_files = [f for f in os.listdir(folder_path) if f.lower().endswith('.pdf')]
    print(f'\nFolder: {folder_name} ({len(pdf_files)} PDF)')

    for pdf_file in sorted(pdf_files):
        pdf_path = os.path.join(folder_path, pdf_file)
        text = extract_text_from_pdf(pdf_path)

        if not text:
            print(f'  [SKIP] {pdf_file} - teks kosong')
            continue

        # Deteksi arah dari isi teks (opsional)
        arah = arah_default
        text_lower = text.lower()
        if 'surat masuk' in text_lower or 'kepada yth' in text_lower:
            arah = 'Masuk'
        elif 'surat keluar' in text_lower or 'dari:' in text_lower:
            arah = 'Keluar'

        data.append({
            'file': pdf_file,
            'text': text,
            'arah': arah,
            'jenis': jenis,
        })
        print(f'  [OK] {pdf_file} -> {jenis} ({arah})')

df = pd.DataFrame(data)
print(f'\n{"=" * 60}')
print(f'Total PDF terbaca: {len(df)}')
print(f'\nDistribusi Jenis:')
print(df['jenis'].value_counts().to_string())
print(f'\nDistribusi Arah:')
print(df['arah'].value_counts().to_string())

## 3. Cek Data
Pastikan setiap kategori punya minimal 5 dokumen. Kalau kurang, tambahkan PDF.

In [ ]:
# Validasi data
print('Validasi dataset:')
print('=' * 60)
for jenis, count in df['jenis'].value_counts().items():
    status = 'OK' if count >= 5 else 'KURANG (min 5)'
    print(f'  {jenis:25s} : {count:3d} dokumen  [{status}]')

if len(df) < 20:
    print(f'\n[WARNING] Total data terlalu sedikit ({len(df)}). Disarankan minimal 50 dokumen.')
else:
    print(f'\n[OK] Total data cukup: {len(df)} dokumen')

## 4. Preprocessing Teks

In [ ]:
stemmer_factory = StemmerFactory()
stemmer = stemmer_factory.createStemmer()

stopword_factory = StopWordRemoverFactory()
stopword_list = stopword_factory.getStopWords()

custom_stopwords = set(stopword_list) | {
    'pt', 'cv', 'tbk', 'abt', 'vi', '2025',
    'nomor', 'perihal', 'lampiran', 'kepada', 'yth',
    'yang', 'dan', 'di', 'dengan', 'untuk', 'pada', 'dari',
    'ini', 'itu', 'adalah', 'ke', 'oleh', 'sebagai', 'juga',
    'akan', 'telah', 'sudah', 'atau', 'dalam', 'tidak',
    'ada', 'dapat', 'bisa', 'lebih',
}

def preprocess_text(text):
    text = text.lower()
    text = re.sub(r'[^a-z\s]', ' ', text)
    text = re.sub(r'\s+', ' ', text).strip()
    tokens = text.split()
    tokens = [t for t in tokens if t not in custom_stopwords and len(t) > 2]
    tokens = stemmer.stem(' '.join(tokens)).split()
    return ' '.join(tokens)

df['clean_text'] = df['text'].apply(preprocess_text)

print('Contoh hasil preprocessing:')
print('=' * 60)
for i in range(min(3, len(df))):
    print(f'\nFile    : {df.iloc[i]["file"]}')
    print(f'Original: {df.iloc[i]["text"][:100]}...')
    print(f'Cleaned : {df.iloc[i]["clean_text"][:100]}...')
    print(f'Jenis   : {df.iloc[i]["jenis"]}  |  Arah: {df.iloc[i]["arah"]}')

## 5. Ekstraksi Fitur TF-IDF

In [ ]:
tfidf_vectorizer = TfidfVectorizer(
    max_features=5000,
    ngram_range=(1, 2),
    min_df=1,
    max_df=0.95
)

X = tfidf_vectorizer.fit_transform(df['clean_text'])
y_arah = df['arah']
y_jenis = df['jenis']

print(f'Shape matriks TF-IDF: {X.shape}')
print(f'Jumlah fitur: {len(tfidf_vectorizer.get_feature_names_out())}')
print()
print('Top 20 fitur TF-IDF:')
feature_names = tfidf_vectorizer.get_feature_names_out()
tfidf_sum = X.sum(axis=0).A1
top_indices = tfidf_sum.argsort()[-20:][::-1]
for idx in top_indices:
    print(f'  {feature_names[idx]:25s} -> bobot: {tfidf_sum[idx]:.4f}')

## 6. Split Data (80% Training, 20% Testing)

In [ ]:
# Cek apakah ada cukup data untuk stratified split
min_samples = y_jenis.value_counts().min()
test_size = 0.2 if min_samples >= 5 else 0.1

X_train_arah, X_test_arah, y_train_arah, y_test_arah = train_test_split(
    X, y_arah, test_size=test_size, random_state=42, stratify=y_arah
)

X_train_jenis, X_test_jenis, y_train_jenis, y_test_jenis = train_test_split(
    X, y_jenis, test_size=test_size, random_state=42, stratify=y_jenis
)

print(f'Total data     : {X.shape[0]} dokumen')
print(f'Training ({int((1-test_size)*100)}%) : {X_train_arah.shape[0]} dokumen')
print(f'Testing ({int(test_size*100)}%)  : {X_test_arah.shape[0]} dokumen')
print()
print('Distribusi Arah (Training):')
print(y_train_arah.value_counts().to_string())
print()
print('Distribusi Jenis (Training):')
print(y_train_jenis.value_counts().to_string())

## 7. Training Model - Klasifikasi Arah Dokumen

In [ ]:
model_arah = MultinomialNB(alpha=1.0)
model_arah.fit(X_train_arah, y_train_arah)

y_pred_arah = model_arah.predict(X_test_arah)

print('KLASIFIKASI ARAH DOKUMEN (Masuk/Keluar)')
print('=' * 50)
print(f'Accuracy : {accuracy_score(y_test_arah, y_pred_arah):.4f}')
print(f'Precision: {precision_score(y_test_arah, y_pred_arah, average="weighted"):.4f}')
print(f'Recall   : {recall_score(y_test_arah, y_pred_arah, average="weighted"):.4f}')
print(f'F1-Score : {f1_score(y_test_arah, y_pred_arah, average="weighted"):.4f}')
print()
print('Classification Report:')
print(classification_report(y_test_arah, y_pred_arah))

## 8. Confusion Matrix - Arah Dokumen

In [ ]:
cm_arah = confusion_matrix(y_test_arah, y_pred_arah, labels=model_arah.classes_)

plt.figure(figsize=(8, 6))
sns.heatmap(cm_arah, annot=True, fmt='d', cmap='Blues',
            xticklabels=model_arah.classes_, yticklabels=model_arah.classes_,
            linewidths=0.5, linecolor='gray')
plt.title('Confusion Matrix - Klasifikasi Arah Dokumen', fontsize=14, fontweight='bold')
plt.xlabel('Prediksi', fontsize=12)
plt.ylabel('Aktual', fontsize=12)
plt.tight_layout()
plt.savefig('confusion_matrix_arah.png', dpi=150, bbox_inches='tight')
plt.show()
print('Gambar disimpan: confusion_matrix_arah.png')

## 9. Training Model - Klasifikasi Jenis Dokumen

In [ ]:
model_jenis = MultinomialNB(alpha=1.0)
model_jenis.fit(X_train_jenis, y_train_jenis)

y_pred_jenis = model_jenis.predict(X_test_jenis)

print('KLASIFIKASI JENIS DOKUMEN')
print('=' * 50)
print(f'Accuracy : {accuracy_score(y_test_jenis, y_pred_jenis):.4f}')
print(f'Precision: {precision_score(y_test_jenis, y_pred_jenis, average="weighted"):.4f}')
print(f'Recall   : {recall_score(y_test_jenis, y_pred_jenis, average="weighted"):.4f}')
print(f'F1-Score : {f1_score(y_test_jenis, y_pred_jenis, average="weighted"):.4f}')
print()
print('Classification Report:')
print(classification_report(y_test_jenis, y_pred_jenis))

## 10. Confusion Matrix - Jenis Dokumen

In [ ]:
cm_jenis = confusion_matrix(y_test_jenis, y_pred_jenis, labels=model_jenis.classes_)

plt.figure(figsize=(10, 8))
sns.heatmap(cm_jenis, annot=True, fmt='d', cmap='Oranges',
            xticklabels=model_jenis.classes_, yticklabels=model_jenis.classes_,
            linewidths=0.5, linecolor='gray')
plt.title('Confusion Matrix - Klasifikasi Jenis Dokumen', fontsize=14, fontweight='bold')
plt.xlabel('Prediksi', fontsize=12)
plt.ylabel('Aktual', fontsize=12)
plt.xticks(rotation=45, ha='right')
plt.yticks(rotation=0)
plt.tight_layout()
plt.savefig('confusion_matrix_jenis.png', dpi=150, bbox_inches='tight')
plt.show()
print('Gambar disimpan: confusion_matrix_jenis.png')

## 11. Cross-Validation (5-Fold)

In [ ]:
n_folds = min(5, y_jenis.value_counts().min())

cv_arah = cross_val_score(MultinomialNB(alpha=1.0), X, y_arah, cv=n_folds, scoring='accuracy')
cv_jenis = cross_val_score(MultinomialNB(alpha=1.0), X, y_jenis, cv=n_folds, scoring='accuracy')

print(f'Hasil {n_folds}-Fold Cross-Validation:')
print('=' * 50)
print(f'\nArah Dokumen:')
for i, score in enumerate(cv_arah, 1):
    print(f'  Fold {i}: {score:.4f}')
print(f'  Mean : {cv_arah.mean():.4f}')
print(f'  Std  : {cv_arah.std():.4f}')

print(f'\nJenis Dokumen:')
for i, score in enumerate(cv_jenis, 1):
    print(f'  Fold {i}: {score:.4f}')
print(f'  Mean : {cv_jenis.mean():.4f}')
print(f'  Std  : {cv_jenis.std():.4f}')

fig, axes = plt.subplots(1, 2, figsize=(12, 5))
axes[0].bar(range(1, n_folds+1), cv_arah, color='#3B82F6', alpha=0.8)
axes[0].axhline(y=cv_arah.mean(), color='red', linestyle='--', label=f'Mean: {cv_arah.mean():.4f}')
axes[0].set_title('Cross-Validation - Arah Dokumen', fontweight='bold')
axes[0].set_xlabel('Fold'); axes[0].set_ylabel('Accuracy'); axes[0].set_ylim(0, 1.1); axes[0].legend()

axes[1].bar(range(1, n_folds+1), cv_jenis, color='#F59E0B', alpha=0.8)
axes[1].axhline(y=cv_jenis.mean(), color='red', linestyle='--', label=f'Mean: {cv_jenis.mean():.4f}')
axes[1].set_title('Cross-Validation - Jenis Dokumen', fontweight='bold')
axes[1].set_xlabel('Fold'); axes[1].set_ylabel('Accuracy'); axes[1].set_ylim(0, 1.1); axes[1].legend()

plt.tight_layout()
plt.savefig('cross_validation.png', dpi=150, bbox_inches='tight')
plt.show()
print('Gambar disimpan: cross_validation.png')

## 12. Simpan Model (untuk backend)

In [ ]:
import joblib

MODEL_DIR = '../backend/ml_model'
os.makedirs(MODEL_DIR, exist_ok=True)

joblib.dump(model_arah, os.path.join(MODEL_DIR, 'arah_pipeline.pkl'))
joblib.dump(model_jenis, os.path.join(MODEL_DIR, 'jenis_pipeline.pkl'))
joblib.dump(tfidf_vectorizer, os.path.join(MODEL_DIR, 'tfidf_vectorizer.pkl'))

print('Model berhasil disimpan!')
print(f'  -> {MODEL_DIR}/arah_pipeline.pkl')
print(f'  -> {MODEL_DIR}/jenis_pipeline.pkl')
print(f'  -> {MODEL_DIR}/tfidf_vectorizer.pkl')
print(f'\nModel siap dipakai di backend. Restart server untuk memuat model baru.')